# Data exploration

Sanity-check the BTS ingest. Run after `data.ingest.download_db1b` and `data.ingest.download_ontime` complete.

What we expect to see for Spirit (NK):
- 2020 Q2: COVID crater
- 2021-2022: strong recovery
- 2024: visible decline in route count and elevated cancellations (GTF + restructuring)

In [ ]:
import duckdb
from data.paths import ONTIME, DB1B_MARKET

conn = duckdb.connect(':memory:')
ontime_glob = f'{ONTIME}/year=*/month=*/data.parquet'
db1b_glob = f'{DB1B_MARKET}/year=*/quarter=*/data.parquet'

## Spirit quarterly: routes, departures, cancellations

In [ ]:
spirit = conn.execute(f"""
    SELECT YEAR, QUARTER,
           COUNT(DISTINCT ORIGIN || '-' || DEST) AS routes,
           COUNT(*) AS scheduled,
           SUM(CASE WHEN CANCELLED < 0.5 THEN 1 ELSE 0 END) AS performed,
           SUM(CASE WHEN CANCELLED >= 0.5 THEN 1 ELSE 0 END) AS cancelled,
           1.0 * SUM(CASE WHEN CANCELLED >= 0.5 THEN 1 ELSE 0 END) / COUNT(*) AS cancel_rate
    FROM read_parquet('{ontime_glob}')
    WHERE MARKETING_IATA = 'NK'
    GROUP BY YEAR, QUARTER ORDER BY YEAR, QUARTER
""").pl()
spirit

## Spirit DB1B passenger sample by quarter

In [ ]:
conn.execute(f"""
    SELECT YEAR, QUARTER,
           SUM(PASSENGERS) AS pax_sample,
           AVG(MARKET_FARE) FILTER (WHERE MARKET_FARE > 0 AND BULK_FARE = 0) AS avg_fare
    FROM read_parquet('{db1b_glob}')
    WHERE TICKET_CARRIER = 'NK'
    GROUP BY YEAR, QUARTER ORDER BY YEAR, QUARTER
""").pl()

## ULCC cancellation rates 2023 vs 2024 (Spirit vs Frontier vs Allegiant)

In [ ]:
conn.execute(f"""
    SELECT YEAR, MARKETING_IATA,
           COUNT(*) AS scheduled,
           1.0 * SUM(CASE WHEN CANCELLED >= 0.5 THEN 1 ELSE 0 END) / COUNT(*) AS cancel_rate
    FROM read_parquet('{ontime_glob}')
    WHERE MARKETING_IATA IN ('NK', 'F9', 'G4') AND YEAR IN (2023, 2024)
    GROUP BY YEAR, MARKETING_IATA
    ORDER BY MARKETING_IATA, YEAR
""").pl()

## Spirit top routes lost between 2023 Q1 and 2024 Q3

In [ ]:
conn.execute(f"""
    WITH q23 AS (
      SELECT ORIGIN, DEST, COUNT(*) AS deps_23
      FROM read_parquet('{ontime_glob}')
      WHERE MARKETING_IATA='NK' AND YEAR=2023 AND MONTH IN (1,2,3) AND CANCELLED<0.5
      GROUP BY ORIGIN, DEST
    ),
    q24 AS (
      SELECT ORIGIN, DEST, COUNT(*) AS deps_24
      FROM read_parquet('{ontime_glob}')
      WHERE MARKETING_IATA='NK' AND YEAR=2024 AND MONTH IN (7,8,9) AND CANCELLED<0.5
      GROUP BY ORIGIN, DEST
    )
    SELECT q23.ORIGIN, q23.DEST, deps_23, COALESCE(deps_24, 0) AS deps_24
    FROM q23 LEFT JOIN q24 USING (ORIGIN, DEST)
    WHERE COALESCE(deps_24, 0) = 0
    ORDER BY deps_23 DESC LIMIT 20
""").pl()